# Research Agent Demo

This notebook demonstrates the LangGraph-based Research Agent that enriches partner context with:
1. **Internal Sales Data** - Retrieved from Confluent Sales Cloud Excel file
2. **External Context** - Retrieved via Tavily web search
3. **Enriched Partner Profile** - Synthesized insights and recommendations

## Architecture

The Research Agent uses a LangGraph workflow with three main nodes:
- `research_internal`: Retrieves and analyzes internal sales history
- `research_external`: Searches web for partner background and technology alignment
- `synthesize_profile`: Uses LLM to create comprehensive partner profile

## Setup

In [1]:
# Import required libraries
import os
from dotenv import load_dotenv
from research_agent import (
    research_partner,
    create_research_agent,
    retrieve_sales_history,
    analyze_partner_maturity,
    search_partner_background
)

# Load environment variables
load_dotenv()

# Verify API keys are set
assert os.getenv("WATSONX_APIKEY"), "WATSONX_APIKEY not found in environment"
assert os.getenv("WATSONX_PROJECT_ID"), "WATSONX_PROJECT_ID not found in environment"
assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not found in environment"

print("✓ Environment setup complete")

✓ Environment setup complete


## Demo 1: Test Individual Tools

Let's first test the individual tools that make up the Research Agent.

### 1.1 Retrieve Sales History

In [2]:
# Test retrieving sales history for Confluent
# Note: When using .invoke() on a tool, it returns the raw function result
sales_data = retrieve_sales_history.invoke({"partner_name": "Confluent"})

print("=" * 80)
print("SALES HISTORY FOR CONFLUENT")
print("=" * 80)

# Check if there was an error
if 'error' in sales_data:
    print(f"\n Error: {sales_data['error']}")
elif not sales_data.get('found', False):
    print(f"\n{sales_data.get('message', 'No data found')}")
else:
    print(f"\nFound: {sales_data['found']}")
    print(f"\nSummary:")
    for key, value in sales_data['summary'].items():
        print(f"  {key}: {value}")

    print(f"\nStakeholders: {', '.join(sales_data['stakeholders'])}")
    print(f"\nProducts Used: {', '.join(sales_data['products_used'])}")
    print(f"\nSales Velocity: {sales_data['sales_velocity']}")

    print(f"\nOpportunities ({len(sales_data['opportunities'])}):")
    for opp in sales_data['opportunities'][:3]:  # Show first 3
        print(f"\n  - {opp['opportunity_name']}")
        print(f"    Stage: {opp['stage']} | Amount: ${opp['amount']:,}")
        print(f"    Products: {opp['products']}")
        print(f"    Next Steps: {opp['next_steps']}")

DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Products', 'Next Steps']
DEBUG - First few rows:
0       Opportunity Name Owner Full Name Stage   Amount           Close Date  \
0       Confluent Cognos    Kylie Brittz   Won  1000000  2023-05-30 00:00:00   
1  Confluent watsonx ESA       Anand Das   Won   250000  2025-01-30 00:00:00   

0                                           Products  \
0                                             Cognos   
1  watsonx Orchestrate, watsonx.governance, watso...   

0                  Next Steps  
0          Deal signed by CPO  
1  CFO signed 1 year renewals  
SALES HISTORY FOR CONFLUENT

Found: True

Summary:
  total_opportunities: 9
  won_deals: 4
  lost_deals: 2
  active_deals: 3
  total_won_amount: 1900000
  total_active_amount: 4000000
  win_rate: 44.4%

Stakeholders: Anand Das, Kylie Brittz

Products Used: watsonx.governance, Cognos, watsonx.ai, watsonx.data, watsonx Orchestrate

Sales Veloc

### 1.2 Analyze Partner Maturity

In [3]:
# Analyze partner maturity based on sales data
if 'error' not in sales_data and sales_data.get('found', False):
    maturity = analyze_partner_maturity.invoke({"sales_data": sales_data})

    print("=" * 80)
    print("PARTNER MATURITY ANALYSIS")
    print("=" * 80)
    print(f"\nMaturity Level: {maturity['maturity_level']}")
    print(f"Engagement Score: {maturity['engagement_score']}/100")
    print(f"\nAssessment: {maturity['assessment']}")
    print(f"\nRecommendation: {maturity['recommendation']}")
else:
    print("⚠️  Skipping maturity analysis - no sales data available")

PARTNER MATURITY ANALYSIS

Maturity Level: Strategic Partner
Engagement Score: 100/100

Assessment: Established relationship with multiple successful deals

Recommendation: Focus on expansion and upsell opportunities


### 1.3 Search External Context

In [4]:
# Search for partner background (this will make a real web search)
background = search_partner_background.invoke({"partner_name": "Confluent"})

print("=" * 80)
print("EXTERNAL PARTNER BACKGROUND")
print("=" * 80)
print(background[:500] + "..." if len(background) > 500 else background)

EXTERNAL PARTNER BACKGROUND
{'query': 'Confluent company background recent news technology partnerships', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://www.confluent.io/press-releases/', 'title': 'Press Releases - Confluent', 'content': "Confluent Announces Strategic Partnership with Jio Platforms Limited to Accelerate India's Development of GenAI and Next-Gen Applications with Data Streams", 'score': 0.72605723, 'raw_content': None}, {'url': 'https://www.crn.com/news/channel-news/2026/confluent-looks-to-streamline-partner-resell-efforts-with-new-initiative', 'title': 'Confluent Looks To Streamline Partner Resell Efforts With New ...', 'content': 'Confluent has launched a new reseller partner program that the data streaming platform developer says will help system integrator and solution', 'score': 0.5855047, 'raw_content': None}, {'url': 'https://newsroom.ibm.com/2025-12-08-ibm-to-acquire-confluent-to-create-smart-data-platform-for-enterprise-

## Demo 2: Full Research Agent Workflow

Now let's run the complete Research Agent workflow that combines all steps.

In [ ]:
# Research Confluent partner
profile = research_partner("Confluent")

print("=" * 80)
print(f"ENRICHED PARTNER PROFILE: {profile['partner_name']}")
print("=" * 80)
print(f"\nMaturity Level: {profile['maturity_level']}")
print(f"Sales Velocity: {profile['sales_velocity']}")

print(f"\n{'=' * 80}")
print("DEAL BLOCKERS")
print(f"{'=' * 80}")
if profile['deal_blockers']:
    for blocker in profile['deal_blockers']:
        print(f"\n  • {blocker['opportunity']}")
        print(f"    Reason: {blocker['reason']}")
else:
    print("No deal blockers identified")

print(f"\n{'=' * 80}")
print("AI-GENERATED SYNTHESIS")
print(f"{'=' * 80}")
print(profile['synthesis'])

DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Products', 'Next Steps']
DEBUG - First few rows:
0       Opportunity Name Owner Full Name Stage   Amount           Close Date  \
0       Confluent Cognos    Kylie Brittz   Won  1000000  2023-05-30 00:00:00   
1  Confluent watsonx ESA       Anand Das   Won   250000  2025-01-30 00:00:00   

0                                           Products  \
0                                             Cognos   
1  watsonx Orchestrate, watsonx.governance, watso...   

0                  Next Steps  
0          Deal signed by CPO  
1  CFO signed 1 year renewals  


## Demo 3: Visualize the LangGraph Workflow

In [ ]:
# Create the agent and visualize the graph
agent = create_research_agent()

# Display the graph structure
try:
    from IPython.display import Image, display
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not display graph: {e}")
    print("\nGraph structure:")
    print("START -> research_internal -> research_external -> synthesize_profile -> END")

## Demo 4: Detailed Internal Data Analysis

In [ ]:
# Deep dive into internal data
internal_data = profile['internal_data']
sales_history = internal_data['sales_history']

print("=" * 80)
print("DETAILED SALES ANALYSIS")
print("=" * 80)

# Opportunities by stage
print("\nOpportunities by Stage:")
stages = {}
for opp in sales_history['opportunities']:
    stage = opp['stage']
    stages[stage] = stages.get(stage, 0) + 1
for stage, count in stages.items():
    print(f"  {stage}: {count}")

# Products breakdown
print("\nProducts Engaged:")
for product in sales_history['products_used']:
    print(f"  • {product}")

# Timeline analysis
print("\nRecent Opportunities:")
sorted_opps = sorted(
    sales_history['opportunities'],
    key=lambda x: x['close_date'],
    reverse=True
)[:5]
for opp in sorted_opps:
    print(f"  {opp['close_date']}: {opp['opportunity_name']} ({opp['stage']})")

## Demo 5: Research Different Partner

Test the agent with a partner that may not exist in the database.

In [ ]:
# Research a new/unknown partner
new_partner_profile = research_partner("Acme Corporation")

print("=" * 80)
print(f"ENRICHED PARTNER PROFILE: {new_partner_profile['partner_name']}")
print("=" * 80)
print(f"\nMaturity Level: {new_partner_profile['maturity_level']}")
print(f"Sales Velocity: {new_partner_profile['sales_velocity']}")
print(f"\n{'=' * 80}")
print("SYNTHESIS")
print(f"{'=' * 80}")
print(new_partner_profile['synthesis'])

## Demo 6: Export Profile for Supervisory Agent

Format the enriched profile for consumption by a Supervisory Agent.

In [ ]:
import json

# Create a structured output for supervisory agent
supervisory_input = {
    "partner_name": profile['partner_name'],
    "maturity_level": profile['maturity_level'],
    "sales_velocity": profile['sales_velocity'],
    "engagement_score": profile['internal_data']['maturity_analysis']['engagement_score'],
    "deal_blockers": profile['deal_blockers'],
    "key_stakeholders": profile['internal_data']['sales_history']['stakeholders'],
    "products_used": profile['internal_data']['sales_history']['products_used'],
    "recommendation": profile['internal_data']['maturity_analysis']['recommendation'],
    "external_signals_summary": profile['synthesis'][:300] + "..."
}

print("=" * 80)
print("OUTPUT FOR SUPERVISORY AGENT")
print("=" * 80)
print(json.dumps(supervisory_input, indent=2))

## Summary

The Research Agent successfully:

1. ✅ **Retrieves Internal Sales Data** from Excel file
   - Prior opportunities and stage history
   - Previous follow-ups and next steps
   - Known stakeholders and products

2. ✅ **Retrieves External Context** via Tavily web search
   - Partner company background
   - Recent announcements and news
   - Technology alignment signals

3. ✅ **Returns Enriched Partner Profile** with:
   - Partner maturity level assessment
   - Historical sales velocity metrics
   - Prior deal blockers and challenges
   - Relevant external market signals
   - AI-synthesized recommendations

The agent is ready to be integrated with a Supervisory Agent for multi-agent workflows!